In [16]:
import json
import sys
from hashlib import sha256
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if Path(sys.executable).resolve() != (root / ".venv/Scripts/python.exe").resolve():
    raise RuntimeError("Select the Cyberion (.venv) kernel.")

dataset = "DS001"
meta = json.loads((root / f"docs/{dataset}-inventory.json").read_text())
raw = (root / meta["file"]).read_bytes()
if sha256(raw).hexdigest().upper() != meta["sha256"]:
    raise ValueError("Raw-file hash mismatch.")

records = [
    json.loads(line)
    for line in raw.decode("utf-8-sig").splitlines()
    if line.strip()
]
if len(records) != meta["records"]:
    raise ValueError("Record count differs from the inventory.")
print(f"{dataset}: {len(records)} records; hash verified.")

DS001: 1127 records; hash verified.


In [17]:
from collections import Counter

sources = (
    "Microsoft-Windows-Sysmon",
    "Microsoft-Windows-Security-Auditing",
    "Microsoft-Windows-PowerShell",
)
report = {"dataset_id": dataset, "sha256": meta["sha256"], "sources": {}}

for source in sources:
    rows = [record for record in records if record.get("SourceName") == source]
    event_ids = dict(Counter(str(row["EventID"]) for row in rows))
    fields = {}
    for field in sorted({key for row in rows for key in row}):
        values = [
            row[field] for row in rows
            if row.get(field) not in (None, "", "-")
        ]
        fields[field] = {
            "populated": len(values),
            "types": sorted({type(value).__name__ for value in values}),
        }

    report["sources"][source] = {
        "records": len(rows), "event_ids": event_ids, "fields": fields,
    }
    print(f"\n{source}: {len(rows)} records; Event IDs: {event_ids}")
    for field, stats in fields.items():
        print(f"{field}: {stats['populated']} populated; {', '.join(stats['types'])}")

(root / f"docs/{dataset}-fields.json").write_text(
    json.dumps(report, indent=2) + "\n", encoding="utf-8"
)


Microsoft-Windows-Sysmon: 892 records; Event IDs: {'10': 184, '12': 401, '13': 48, '7': 238, '1': 3, '9': 6, '5': 1, '18': 1, '11': 5, '2': 1, '23': 3, '17': 1}
@timestamp: 892 populated; str
Archived: 3 populated; str
CallTrace: 184 populated; str
Channel: 892 populated; str
CommandLine: 3 populated; str
Company: 238 populated; str
CreationUtcTime: 6 populated; str
CurrentDirectory: 3 populated; str
Description: 238 populated; str
Details: 48 populated; str
Device: 6 populated; str
EventID: 892 populated; int
EventType: 451 populated; str
FileVersion: 238 populated; str
GrantedAccess: 184 populated; str
Hashes: 244 populated; str
Hostname: 892 populated; str
Image: 708 populated; str
ImageLoaded: 238 populated; str
IntegrityLevel: 3 populated; str
IsExecutable: 3 populated; str
Keywords: 892 populated; str
Level: 892 populated; str
LogonGuid: 3 populated; str
LogonId: 3 populated; str
Message: 892 populated; str
OriginalFileName: 238 populated; str
ParentCommandLine: 3 populated; str

13878

In [18]:
if dataset == "DS002":
    powershell = [
        row for row in records
        if row.get("SourceName") == "Microsoft-Windows-PowerShell"
    ]
    print("Flags:", dict(Counter(
        row.get("ERROR_EVT_UNRESOLVED", "<absent>") for row in powershell
    )))
    for row in powershell:
        if row["EventID"] == 4104:
            print({field: row.get(field) for field in (
                "RecordNumber", "ScriptBlockId", "MessageNumber", "MessageTotal"
            )})

In [19]:
for dataset in ("DS001", "DS002"):
    report = json.loads(
        (root / f"docs/{dataset}-normalization.json").read_text()
    )
    raw = (root / report["source_file"]).read_bytes()
    output = (root / report["output_file"]).read_bytes()

    assert sha256(raw).hexdigest().upper() == report["raw_sha256"], "Raw hash mismatch"
    assert sha256(output).hexdigest().upper() == report["output_sha256"], "Output hash mismatch"

    with (root / report["source_file"]).open(encoding="utf-8-sig") as stream:
        originals = {
            number: json.loads(line)
            for number, line in enumerate(stream, 1)
            if line.strip()
        }

    rows = [json.loads(line) for line in output.splitlines()]
    locations = [row["record_locator"] for row in rows]
    assert len(locations) == len(set(locations)), "Repeated source-line reference"
    assert set(locations) == set(originals), "Missing or unexpected source lines"

    for row in rows:
        number = row["record_locator"]
        assert row["raw"] == originals[number], f"Raw content mismatch at line {number}"
        assert row["event_uid"] == f"{dataset}:{report['raw_sha256']}:{number}", "UID mismatch"

    blocks = [
        row for row in rows
        if row["provider"] == "Microsoft-Windows-PowerShell"
        and row["event_id"] == 4104
    ]
    assert len(blocks) == (1 if dataset == "DS002" else 0), "Unexpected script-block count"
    for row in blocks:
        original = row["raw"]
        assert (original["MessageNumber"], original["MessageTotal"]) == ("1", "1"), "Fragment review needed"
        assert row["script_block_id"] == original["ScriptBlockId"], "Script ID mismatch"
        assert row["script_block_text"] == original["ScriptBlockText"], "Script text mismatch"
        assert row["script_block_text"].strip(), "Empty script block"

    result = {
        "dataset_id": dataset,
        "status": "PASS",
        "raw_sha256": report["raw_sha256"],
        "output_sha256": report["output_sha256"],
        "traceable_records": len(rows),
        "single_fragment_script_blocks": len(blocks),
    }
    (root / f"docs/{dataset}-validation.json").write_text(
        json.dumps(result, indent=2) + "\n", encoding="utf-8"
    )
    print(f"{dataset}: {len(rows)} records traced; {len(blocks)} script blocks verified.")

DS001: 1127 records traced; 0 script blocks verified.
DS002: 4348 records traced; 1 script blocks verified.
